# Building sections — Practical How-To

This notebook shows how to create and inspect a schematic building section, compare two usage layouts, and vary PV capacity directly in Python.

We build a demonstration project by hand using all seven usage categories in the current schema. The figures are static SVGs. Area checks and reproducibility are collected in the technical notes at the end.

## 0. Setup

Use the same Python environment as for the other `pexl` notebooks. Run the cells in order. The SVG renderer uses the standard library; no notebook widgets are needed.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML, SVG, display

from pexl import Project
from pexl.spatial import section_from_scenario, render_svg

## 1. Create a demonstration project

A `Project` contains scenarios, whose values are assigned through `scenario.v`. This example includes residential, office, primary and secondary education, food retail, other retail and other uses.

We supply five buildings and an average of 5.5 storeys as the starting point. The values are illustrative. The default allocation can adapt individual building widths and heights while preserving building count and usage areas.

In [ ]:
project = Project()
scenario = project.get_or_create_scenario("Demonstration | Mixed-use buildings")
scenario.v.project_name = "Demonstration"
scenario.v.project_scenario_name = "Mixed-use buildings"

usage_areas = {
    "NFA_office": 1350.0,
    "NFA_retailfood": 600.0,
    "NFA_retailother": 450.0,
    "NFA_schoolprim": 600.0,
    "NFA_schoolsec": 2000.0,
    "NFA_other": 450.0,
    "NFA_residential": 5000.0,
}
for name, area in usage_areas.items():
    setattr(scenario.v, name, area)

scenario.v.NFA_total = sum(usage_areas.values())
scenario.v.building_count = 5
scenario.v.storey_count_avg = 5.5
scenario.v.rh_avg = 3.0
scenario.v.rh_ceiling = 0.35
scenario.v.hull_transmittance_walls = 0.30
scenario.v.hull_transmittance_roof = 0.20
scenario.v.PV_is_used = 1
scenario.v.PV_m2_per_kWp = 5.0
scenario.v.PV_kWp = 120.0
scenario.v.PV_module_area = scenario.v.PV_kWp * scenario.v.PV_m2_per_kWp

scenario

## 2. Generate and inspect the default section

`section_from_scenario` generates geometry; `render_svg` creates a static figure. `layout="default"` places supermarket and other retail at ground level, then assigns each school and office usage to the least occupied building as one continuous block. Blocks can extend upwards.

After each block, available “other” area tops up its last floor. When that buffer is exhausted, residential area fills the gap before any new residential storeys are added. Office can use an otherwise unassigned ground floor; remaining residential area follows above the existing uses. Residential space is distributed across the lowest building tops to balance height differences where possible. Preserving continuous usage blocks takes priority over equal heights.

The horizontal axis uses 15 m intervals; floor labels are shown on the left. Horizontal and vertical scales are independent. Slabs end flush with the green construction face; lilac insulation runs continuously outside the slab edges.

In [ ]:
section = section_from_scenario(scenario, seed=42, layout="default")
section

In [ ]:

svg = render_svg(section)

def show_section(svg):
    display(HTML('<div style="overflow-x:auto;background:white">' + svg + '</div>'))

show_section(svg)

The section is an ordinary dictionary. Inspect the spaces and the amount of PV that could be placed before using the figure in a report.

In [ ]:
spaces = pd.DataFrame(section["spaces"])
display(spaces[["floor_id", "usage", "area_m2", "x_m", "width_m"]])
display(pd.Series({
    "layout": section["layout"],
    "buildings": len(section["buildings"]),
    "occupied floors": len(section["floors"]),
    "PV module area [m²]": section["pv"]["requested_area_m2"],
    "unplaced PV [m²]": section["pv"]["unplaced_area_m2"],
}))

## 3. Compare allocation modes

`separateResidential` preserves the earlier layout: special uses concentrate in the first building, offices fill available lower floors across buildings, and residential fills the remainder building by building. Use this to compare the new default with the previous result.

In [ ]:
separate = section_from_scenario(scenario, layout="separateResidential")
show_section(render_svg(separate))

`equalspread` distributes each usage equally among buildings, with offices below and residential above the intermediate usages. It remains available as a second alternative.

In [ ]:
equal_section = section_from_scenario(scenario, seed=42, layout="equalspread")
show_section(render_svg(equal_section))

## 4. Change PV capacity in a notebook cell

Use an alias for `scenario.v`, assign a new capacity, then regenerate the section. `PV_kWp` is the existing schema variable. Set `PV_module_area` to `None` so the generator derives module area from capacity and `PV_m2_per_kWp` instead of retaining the original explicit area.

These assignments do not run the Excel calculations or update energy results. Any PV area that cannot fit is retained as an unplaced remainder.

In [ ]:
v = scenario.v
v.PV_kWp = 180.0
v.PV_module_area = None

section = section_from_scenario(scenario, layout="default")
svg = render_svg(section)
show_section(svg)

For a small parameter sweep, change the same input in a loop and collect the placement results. Restore the original capacity afterwards.

In [ ]:
original_capacity = v.PV_kWp
rows = []
for capacity in [60.0, 120.0, 240.0, 400.0]:
    v.PV_kWp = capacity
    trial = section_from_scenario(scenario)
    pv = trial["pv"]
    rows.append({
        "PV_kWp": capacity,
        "module_area_m2": pv["requested_area_m2"],
        "placed_m2": sum(p["module_area_m2"] for p in pv["placements"]),
        "unplaced_m2": pv["unplaced_area_m2"],
    })
v.PV_kWp = original_capacity
pd.DataFrame(rows)

## 5. Construction appearance and SVG export

Use `construction="wood"` for light-brown construction. Insulation remains lilac. This option changes the drawing only; it does not assign a material, layer thickness or thermal property in the scenario.

In [ ]:
show_section(render_svg(section, construction="wood"))

output_path = Path("section_demonstration.svg")
# Uncomment to save the default construction appearance:
# output_path.write_text(svg, encoding="utf-8")

## Technical notes

The section is a schematic projection of aggregate inputs, not a reconstruction of measured geometry. Layer thicknesses in the drawing are fixed schematic widths. The common section depth is 15 m and building gaps are 20 m. Geometry generation leaves the scenario unchanged.

In [ ]:
pd.DataFrame(section["assumptions"])

### Area conservation

Width × assumed depth equals NFA. The visible rectangle area (width × room height) does not.

The following comparison checks every usage in the current schema against the generated spaces.

In [ ]:
spaces = pd.DataFrame(section["spaces"])
usage_inputs = pd.DataFrame([
    {"usage": meta.entity_key, "input_m2": value}
    for meta, value in scenario.select(
        domain="usage", measure="area", spatial_scope="NFA", entity_group="usage"
    ).items()
]).set_index("usage")
assert usage_inputs["input_m2"].notna().all(), "Add any new schema usage to the demonstration."
area_check = usage_inputs.join(spaces.groupby("usage")["area_m2"].sum().rename("section_m2"))
area_check["difference_m2"] = area_check["section_m2"] - area_check["input_m2"]
area_check

### Reproducibility

The `default` and `separateResidential` layouts follow fixed allocation rules. The seed affects intermediate usage order in `equalspread`; it does not change area totals. Identical inputs, mode and seed produce the same section.

In [ ]:
assert section == section_from_scenario(scenario, seed=42, layout="default")
a = section_from_scenario(scenario, seed=42, layout="equalspread")
b = section_from_scenario(scenario, seed=7, layout="equalspread")
pd.concat({
    "seed_42_m2": pd.DataFrame(a["spaces"]).groupby("usage")["area_m2"].sum(),
    "seed_7_m2": pd.DataFrame(b["spaces"]).groupby("usage")["area_m2"].sum(),
}, axis=1)

### Quick reference

| Task | Example |
|---|---|
| Generate default layout | `section_from_scenario(scenario)` |
| Earlier residential separation | `section_from_scenario(scenario, layout="separateResidential")` |
| Distribute usages equally | `section_from_scenario(scenario, layout="equalspread")` |
| Render static SVG | `render_svg(section)` |
| Show wood construction | `render_svg(section, construction="wood")` |
| Change PV capacity | `scenario.v.PV_kWp = 180.0` |
| Derive module area from capacity | `scenario.v.PV_module_area = None` |
| Inspect assumptions | `pd.DataFrame(section["assumptions"])` |
| Export SVG | `Path("section.svg").write_text(svg, encoding="utf-8")` |

See [Building sections](../docs/development/building_sections.md) for the dictionary contract and allocation rules.